# Path headers for BM25+ and the reranker (Colab GPU)

Tests whether the context in each document's **path** helps retrieval, before paying to re-embed
the dense index. A path such as `slack/devex/1759999992-opaque-routing-shim-sdk-discuss.json`
becomes the header `Slack · #devex · 2025-10-09 · opaque routing shim sdk discuss`
(`src/retrieval/headers.py`). Slack titles are only the channel name, so the header adds the
thread's topic and date.

Two places get the header. The dense (BGE) index stays as it is:

| | Reranker windows: title only | Reranker windows: header + title |
| --- | --- | --- |
| **BM25+ index: title + content** | A (current) | B |
| **BM25+ index: header + title + content** | C | D |

Each cell uses `bge-reranker-base` at the current setting: 50 candidates, 8 windows of 2,000
characters, 512 tokens, blend weights 1–0.6 with the cross-validated (`blendcv`) result. Also
reported: the plain hybrid without a reranker for both BM25+ indexes, and BM25+ alone.
Only the 300 development questions are used; calibration and final stay untouched.

1. **Runtime → Change runtime type → L4 (or T4) GPU, with High-RAM on.**
2. Run all cells. Expect about 6 minutes for the header index and 30–60 minutes for the two runs.
3. Part 5 downloads `headers_results.zip`. Send it back to add the results to the repository.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
!free -g | head -2

## 1. Drive and project code

Set `WORK` to the Drive folder that holds the data. The defaults match the personal `rag-dense`
layout (`dense_base_titled` at its top level). For the team `files_web_app` folder, use
`DENSE = f"{WORK}/indexes/dense-bge-base"`.

In [ ]:
import json
import os
from google.colab import drive

drive.mount("/content/drive")
WORK = "/content/drive/MyDrive/rag-dense"
CORPUS = f"{WORK}/documents.parquet"
QUESTIONS = f"{WORK}/questions.jsonl"
DRIVE_BM25 = f"{WORK}/indexes/bm25"
DRIVE_BM25_HDR = f"{WORK}/indexes/bm25_headers"  # Saved here after the first build.
DENSE = f"{WORK}/dense_base_titled"  # BGE-base, title on every passage
BM25 = "/content/indexes/bm25"  # Local copies: much faster than reading from Drive.
BM25_HDR = "/content/indexes/bm25_headers"

REPO = "https://github.com/AgenticEnterpriseSystem/Agentic-Enterprise-RAG-System.git"
BRANCH = "hira/dev"
# For a private repository, clone with a GitHub token instead (it is not displayed):
# from getpass import getpass; REPO = f"https://{getpass('GitHub token: ')}@github.com/AgenticEnterpriseSystem/Agentic-Enterprise-RAG-System.git"
if not os.path.exists("/content/project"):
    !git clone --depth 1 -b {BRANCH} {REPO} /content/project
%cd /content/project
!git log -1 --format='%h %s'
!pip install -q -r requirements.txt

## 2. Indexes

Both BM25+ indexes must come from the same `documents.parquet` as the dense index. The plain one
is copied from Drive (or rebuilt, about 5 minutes). The header one is built once (about 6 minutes)
and saved to Drive for later runs.

In [ ]:
import pyarrow.parquet as pq

for path in (CORPUS, QUESTIONS, f"{DENSE}/manifest.json"):
    assert os.path.exists(path), f"Missing {path}"
assert "path" in pq.ParquetFile(CORPUS).schema_arrow.names, "This corpus copy has no path column"
dense_manifest = json.load(open(f"{DENSE}/manifest.json"))
assert dense_manifest.get("complete"), "The dense index is incomplete"
from src.retrieval.corpus_dense import model_label
LABEL = model_label(dense_manifest["model"]["name"])
print("Dense model:", LABEL, dense_manifest["model"]["revision"])

def bm25_matches(path, headers):
    manifest = f"{path}/manifest.json"
    if not os.path.exists(manifest):
        return False
    m = json.load(open(manifest))
    return m["corpus_sha256"] == dense_manifest["corpus_sha256"] and bool(m.get("headers")) == headers

for local, saved, headers in ((BM25, DRIVE_BM25, False), (BM25_HDR, DRIVE_BM25_HDR, True)):
    flag = "--path-headers" if headers else ""
    if not bm25_matches(local, headers):
        if bm25_matches(saved, headers):
            print(f"Copying {saved} from Drive...")
            !rm -rf {local} && mkdir -p /content/indexes && cp -r {saved} {local}
        else:
            print(f"Building {local} (about 5-6 minutes)...")
            !python -m src.retrieval.corpus_bm25 --corpus {CORPUS} --output {local} {flag}
            if headers:
                !mkdir -p {WORK}/indexes && rm -rf {saved} && cp -r {local} {saved}
    assert bm25_matches(local, headers)

from src.retrieval.corpus_bm25 import CorpusBM25
from pathlib import Path
sample = CorpusBM25.load(Path(BM25_HDR), corpus_path=Path(CORPUS))  # Print a few headers.
for i in (0, 1, 200_000, 400_000):
    print(sample.header(sample.doc_ids[i]))
del sample
print("Indexes ready.")

## 3. Development runs

One run per BM25+ index. Each scores the reranker with title-only windows and with header
windows (`--rerank-headers off,on`). The runs write to separate folders so their cost files do not
overwrite each other.

In [ ]:
RERANKER = "bge-reranker-base"
OUT = {"plain": "evaluation/results/headers/bm25_plain", "header": "evaluation/results/headers/bm25_header"}
COMMON = (f"--dense-index {DENSE} --corpus {CORPUS} --questions {QUESTIONS} --split development "
          f"--device cuda --bm25-weights 0.7 --rerank-bm25-weight 0.7 --rerankers {RERANKER} "
          f"--rerank-window-chars 2000 --rerank-max-windows 8 --reranker-max-length 512 "
          f"--reranker-batch-size 32 --rerank-weights 1,0.9,0.8,0.7,0.6 --rerank-headers off,on "
          f"--write-baselines")
for index, out in ((BM25, OUT["plain"]), (BM25_HDR, OUT["header"])):
    !python -m src.evaluation.run_hybrid {COMMON} --bm25-index {index} --output-dir {out}

## 4. Compare

Every row is compared with the current system (cell A without a reranker, the plain hybrid) using
paired 95% bootstrap intervals over the 300 questions. The interval is the range of plausible
average changes; if it includes zero, the change could be chance. The second table repeats
Recall@10 by source type, because headers should help Slack (whose titles are just channel names)
most.

In [ ]:
from collections import defaultdict
from src.evaluation.retrieval_metrics import evaluate_retrieval, ndcg_at_k, paired_bootstrap, recall_at_k

def load_rows(path):
    return [json.loads(line) for line in open(f"{path}/predictions.jsonl")]

P, H = OUT["plain"], OUT["header"]
stem = f"{RERANKER}_maxp2000"
runs = {
    "BM25+ alone (plain index)": f"{P}/bm25_full_development",
    "BM25+ alone (header index)": f"{H}/bm25_bm25hdr_full_development",
    "Hybrid, plain BM25+ (current app)": f"{P}/hybrid_{LABEL}_w0.7_full_development",
    "Hybrid, header BM25+": f"{H}/hybrid_{LABEL}_bm25hdr_w0.7_full_development",
    "A  plain BM25+ / reranker title-only": f"{P}/hybrid_{LABEL}_w0.7_{stem}_blendcv_full_development",
    "B  plain BM25+ / reranker header": f"{P}/hybrid_{LABEL}_w0.7_{stem}_rhdr_blendcv_full_development",
    "C  header BM25+ / reranker title-only": f"{H}/hybrid_{LABEL}_bm25hdr_w0.7_{stem}_blendcv_full_development",
    "D  header BM25+ / reranker header": f"{H}/hybrid_{LABEL}_bm25hdr_w0.7_{stem}_rhdr_blendcv_full_development",
}
rows = {name: load_rows(path) for name, path in runs.items()}
base = rows["Hybrid, plain BM25+ (current app)"]
print(f"{'run':40} {'R@5':>6} {'R@10':>6} {'R@20':>6} {'nDCG@10':>8}   R@10 change vs current app (95% CI; better/worse)   nDCG@10 change")
for name, r in rows.items():
    o = evaluate_retrieval(r)["overall"]
    line = (f"{name:40} {o['recall@5']['mean']:>6.3f} {o['recall@10']['mean']:>6.3f} "
            f"{o['recall@20']['mean']:>6.3f} {o['ndcg@10']['mean']:>8.3f}")
    if r is not base and not name.startswith("BM25+ alone"):
        a, n = paired_bootstrap(r, base, metric=recall_at_k), paired_bootstrap(r, base, metric=ndcg_at_k)
        line += (f"   {a['change']:+.3f} ({a['low']:+.3f} to {a['high']:+.3f}; {a['better']}/{a['worse']})"
                 f"   {n['change']:+.3f} ({n['low']:+.3f} to {n['high']:+.3f})")
    print(line)

print("\nEffect of each header on its own (paired R@10 change, 95% CI):")
for label, new, old in (
        ("BM25+ header, no reranker", "Hybrid, header BM25+", "Hybrid, plain BM25+ (current app)"),
        ("Reranker header, plain BM25+ (B vs A)", "B  plain BM25+ / reranker header", "A  plain BM25+ / reranker title-only"),
        ("BM25+ header, with reranker (C vs A)", "C  header BM25+ / reranker title-only", "A  plain BM25+ / reranker title-only"),
        ("Both headers vs reranker alone (D vs A)", "D  header BM25+ / reranker header", "A  plain BM25+ / reranker title-only")):
    a = paired_bootstrap(rows[new], rows[old], metric=recall_at_k)
    print(f"  {label:42} {a['change']:+.3f} ({a['low']:+.3f} to {a['high']:+.3f}; {a['better']} better, {a['worse']} worse)")

for name, path in runs.items():
    summary = f"{path}/summary.json"
    if "blendcv" in path and os.path.exists(summary):
        print(f"{name[:2]} blend weight chosen per fold:", json.load(open(summary))["cross_validation"]["chosen_per_fold"])

In [ ]:
def by_source(r):
    groups = defaultdict(list)
    for row in r:
        for source in row["source_types"]:
            groups[source].append(row)
    return groups

show = ["Hybrid, plain BM25+ (current app)", "Hybrid, header BM25+",
        "A  plain BM25+ / reranker title-only", "B  plain BM25+ / reranker header",
        "C  header BM25+ / reranker title-only", "D  header BM25+ / reranker header"]
groups = {name: by_source(rows[name]) for name in show}
sources = sorted(groups[show[0]], key=lambda s: -len(groups[show[0]][s]))
print("Recall@10 by source type (questions can name several sources); current = plain hybrid, hyb-hdr = header BM25+ hybrid:")
labels = ["current", "hyb-hdr", "A", "B", "C", "D"]
print(f"{'source':14} {'n':>4} " + " ".join(f"{label:>8}" for label in labels))
for s in sources:
    vals = [evaluate_retrieval(groups[n][s])["overall"]["recall@10"]["mean"] for n in show]
    print(f"{s:14} {len(groups[show[0]][s]):>4} " + " ".join(f"{v:>8.3f}" for v in vals))

slack = lambda name: [r for r in rows[name] if "slack" in r["source_types"]]
print("\nSlack questions, paired R@10 change vs the current app (95% CI):")
for name in show[1:]:
    a = paired_bootstrap(slack(name), slack(show[0]), metric=recall_at_k)
    print(f"  {name:40} {a['change']:+.3f} ({a['low']:+.3f} to {a['high']:+.3f}; {a['better']} better, {a['worse']} worse)")

print("\nReranking cost per question on this GPU:")
for out in OUT.values():
    for stem_name, cost in json.load(open(f"{out}/reranker_costs_development.json")).items():
        print(f"  {stem_name:62} {cost['windows_per_question']:6.1f} windows  ~{cost['seconds_per_question']:.2f}s")

## 5. Save the results

Copies the results to `WORK/results/headers` on Drive and downloads `headers_results.zip`.

In [ ]:
!mkdir -p {WORK}/results && cp -r evaluation/results/headers {WORK}/results/
!cd evaluation/results && zip -qr /content/headers_results.zip headers
from google.colab import files
files.download("/content/headers_results.zip")